In [ ]:
import sqlite3


conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    city TEXT NOT NULL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount REAL NOT NULL,
    order_date TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

-- Insert customers
INSERT INTO customers VALUES
(1, 'Rahul', 'Bengaluru'),
(2, 'Priya', 'Mysuru'),
(3, 'Kiran', 'Tumakuru'),
(4, 'Anu', 'Bengaluru');

-- Insert orders
INSERT INTO orders VALUES
(101, 1, 5000, '2026-01-10'),
(102, 2, 3000, '2026-01-12'),
(103, 1, 7000, '2026-02-05'),
(104, 3, 4000, '2026-02-15'),
(105, 2, 6000, '2026-03-10'),
(106, 1, 2000, '2026-03-20');
""")

def run_query(sql):
    print("\nQUERY RESULT")
    for row in cur.execute(sql):
        print(row)

In [ ]:
run_query("SELECT * FROM customers;")


QUERY RESULT
(1, 'Rahul', 'Bengaluru')
(2, 'Priya', 'Mysuru')
(3, 'Kiran', 'Tumakuru')
(4, 'Anu', 'Bengaluru')


In [ ]:
run_query("SELECT name,city FROM customers WHERE city='Bengaluru';")


QUERY RESULT
('Rahul', 'Bengaluru')
('Anu', 'Bengaluru')


In [ ]:
run_query("SELECT order_id FROM orders ORDER BY amount DESC;")


QUERY RESULT
(103,)
(105,)
(101,)
(104,)
(102,)
(106,)


In [ ]:
run_query("SELECT SUM(amount) AS total_revenue FROM orders;")


QUERY RESULT
(27000.0,)


In [ ]:
run_query("SELECT ROUND(AVG(amount), 2) AS aov FROM orders;")


QUERY RESULT
(4500.0,)


In [ ]:
run_query("SELECT customer_id, COUNT(*) AS total_orders FROM orders GROUP BY customer_id;")


QUERY RESULT
(1, 3)
(2, 2)
(3, 1)


In [ ]:
run_query("SELECT customer_id, SUM(amount) AS revenue FROM orders GROUP BY customer_id HAVING SUM(amount)>8000;")


QUERY RESULT
(1, 14000.0)
(2, 9000.0)


In [ ]:
run_query("SELECT c.name, o.order_id, o.amount FROM customers c INNER JOIN orders o ON c.customer_id=o.customer_id;")


QUERY RESULT
('Rahul', 101, 5000.0)
('Priya', 102, 3000.0)
('Rahul', 103, 7000.0)
('Kiran', 104, 4000.0)
('Priya', 105, 6000.0)
('Rahul', 106, 2000.0)


In [ ]:
run_query("SELECT c.name, COUNT(o.order_id) AS orders FROM customers c LEFT JOIN orders o ON c.customer_id=o.customer_id GROUP BY c.customer_id, c.name;")


QUERY RESULT
('Rahul', 3)
('Priya', 2)
('Kiran', 1)
('Anu', 0)


In [ ]:
run_query("SELECT c.name, SUM(o.amount) AS revenue FROM customers c JOIN orders o ON c.customer_id=o.customer_id GROUP BY c.customer_id, c.name ORDER BY revenue DESC;")


QUERY RESULT
('Rahul', 14000.0)
('Priya', 9000.0)
('Kiran', 4000.0)


In [ ]:
run_query("WITH customer_sales AS ( SELECT customer_id, SUM(amount) AS revenue FROM orders GROUP BY customer_id ) SELECT * FROM customer_sales WHERE revenue>5000;")


QUERY RESULT
(1, 14000.0)
(2, 9000.0)


In [ ]:
run_query("SELECT order_id, customer_id, amount, ROW_NUMBER() OVER( PARTITION BY customer_id ORDER BY amount DESC,order_id ) AS row_num FROM orders;")


QUERY RESULT
(103, 1, 7000.0, 1)
(101, 1, 5000.0, 2)
(106, 1, 2000.0, 3)
(105, 2, 6000.0, 1)
(102, 2, 3000.0, 2)
(104, 3, 4000.0, 1)


In [ ]:
run_query("SELECT order_id, amount, RANK()OVER( ORDER BY amount DESC ) AS sales_rank FROM orders;")


QUERY RESULT
(103, 7000.0, 1)
(105, 6000.0, 2)
(101, 5000.0, 3)
(104, 4000.0, 4)
(102, 3000.0, 5)
(106, 2000.0, 6)


In [ ]:
run_query("SELECT order_date, order_id, amount, SUM(amount)OVER( ORDER BY order_date,order_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW ) AS running_revenue FROM orders;")


QUERY RESULT
('2026-01-10', 101, 5000.0, 5000.0)
('2026-01-12', 102, 3000.0, 8000.0)
('2026-02-05', 103, 7000.0, 15000.0)
('2026-02-15', 104, 4000.0, 19000.0)
('2026-03-10', 105, 6000.0, 25000.0)
('2026-03-20', 106, 2000.0, 27000.0)


In [ ]:
run_query("SELECT substr(order_date, 1,7) AS month, SUM(amount) AS revenue FROM orders GROUP BY substr(order_date,1,7) ORDER BY month;")


QUERY RESULT
('2026-01', 8000.0)
('2026-02', 11000.0)
('2026-03', 8000.0)


In [ ]:
run_query("WITH customer_sales AS ( SELECT customer_id, SUM(amount) AS revenue FROM orders GROUP BY customer_id ) SELECT customer_id, revenue, ROUND( 100.0*revenue/ SUM(revenue)OVER(), 2 ) AS revenue_share_pct FROM customer_sales;")


QUERY RESULT
(1, 14000.0, 51.85)
(2, 9000.0, 33.33)
(3, 4000.0, 14.81)


In [ ]:
run_query("""WITH customer_summary AS (
     SELECT
     c.customer_id,
     c.name,
     c.city,
     COUNT(o.order_id) AS order_count,
     COALESCE(SUM(o.amount),0)
     AS total_spent
     FROM customers c LEFT JOIN orders o ON c.customer_id=o.customer_id GROUP BY
    c.customer_id, c.name, c.city), ranked AS ( SELECT *, RANK()OVER( ORDER BY total_spent DESC ) AS customer_rank FROM customer_summary ) SELECT * FROM ranked ORDER BY customer_rank, customer_id;""")


QUERY RESULT
(1, 'Rahul', 'Bengaluru', 3, 14000.0, 1)
(2, 'Priya', 'Mysuru', 2, 9000.0, 2)
(3, 'Kiran', 'Tumakuru', 1, 4000.0, 3)
(4, 'Anu', 'Bengaluru', 0, 0, 4)


In [ ]:
import pandas as pd

query = """
WITH customer_summary AS (
    SELECT
        c.customer_id,
        c.name,
        c.city,
        COUNT(o.order_id) AS order_count,
        COALESCE(SUM(o.amount), 0) AS total_spent
    FROM customers c
    LEFT JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY
        c.customer_id,
        c.name,
        c.city
)
SELECT *
FROM customer_summary
ORDER BY total_spent DESC
"""

# Uses the restored global connection 'conn'
report = pd.read_sql_query(query, conn)
print(report)

report.to_csv("ecommerce_sql_report.csv", index=False)

   customer_id   name       city  order_count  total_spent
0            1  Rahul  Bengaluru            3      14000.0
1            2  Priya     Mysuru            2       9000.0
2            3  Kiran   Tumakuru            1       4000.0
3            4    Anu  Bengaluru            0          0.0


In [ ]:
conn = sqlite3.connect(":memory:")